# PyTorch — Practical Examples
**CubicNova Data Science Roadmap · Module 10.03**

This notebook moves from tensor basics to autograd, data loading, regression, classification, evaluation, and saving/loading model parameters.

Run from top to bottom. It uses small synthetic datasets so the examples are reproducible and do not require downloading a dataset.

## 0. Imports and device

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

## 1. Tensor basics

In [ ]:
a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
print("Tensor:\n", a)
print("Shape:", a.shape, "dtype:", a.dtype, "device:", a.device)
print("Row sums:", a.sum(dim=1))
print("Column means:", a.mean(dim=0))
print("Reshaped:", a.reshape(3, 2))

Tensors have a shape, dtype, and device. `dim=0` reduces rows to produce column-wise values; `dim=1` reduces columns to produce row-wise values.

## 2. Operations and broadcasting

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
bias = torch.tensor([10.0, 20.0, 30.0])
print("Broadcast addition:\n", x + bias)
print("Element-wise multiply:\n", x * 2)
print("Matrix product:\n", x @ torch.ones((3, 2)))

## 3. Autograd: automatic differentiation

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1
y.backward()
print("y:", y.item())
print("dy/dx:", x.grad.item())  # 2*x + 2 = 8

Autograd tracks operations on tensors that require gradients. Calling `backward()` on a scalar result computes gradients. Parameter gradients accumulate until cleared.

## 4. Dataset and DataLoader

In [ ]:
features = torch.arange(40, dtype=torch.float32).reshape(20, 2)
targets = (features[:, 0] > 18).long()
dataset = TensorDataset(features, targets)
loader = DataLoader(dataset, batch_size=6, shuffle=True)

for batch_x, batch_y in loader:
    print("Batch shapes:", batch_x.shape, batch_y.shape)
    break

A `Dataset` represents examples; a `DataLoader` batches examples and can shuffle them. For a real project, keep training, validation, and test data separate.

## 5. Regression: learn a line

In [ ]:
torch.manual_seed(SEED)
X = torch.linspace(-2, 2, 300).reshape(-1, 1)
y = 2.5 * X - 0.8 + 0.2 * torch.randn_like(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

reg_model = nn.Linear(1, 1).to(device)
reg_loss = nn.MSELoss()
reg_optimizer = torch.optim.SGD(reg_model.parameters(), lr=0.05)
reg_history = []

for epoch in range(100):
    reg_model.train()
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        reg_optimizer.zero_grad()
        pred = reg_model(xb)
        loss = reg_loss(pred, yb)
        loss.backward()
        reg_optimizer.step()
        total += loss.item() * len(xb)
    reg_history.append(total / len(train_loader.dataset))

reg_model.eval()
with torch.inference_mode():
    test_pred = reg_model(X_test.to(device)).cpu()
    test_mse = reg_loss(test_pred, y_test).item()

print("Learned weight:", round(reg_model.weight.item(), 3))
print("Learned bias:", round(reg_model.bias.item(), 3))
print("Test MSE:", round(test_mse, 4))

plt.figure(figsize=(7, 3))
plt.plot(reg_history)
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("Linear regression training")
plt.grid(True, alpha=0.3)
plt.show()

The model learns a weight and bias that approximate the synthetic relationship. Training loss is averaged over examples in each epoch.

## 6. Classification: MLP on make_moons

In [ ]:
X, y = make_moons(n_samples=1200, noise=0.22, random_state=SEED)
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)  # fit on train only
X_test_np = scaler.transform(X_test_np)

X_train_t = torch.tensor(X_train_np, dtype=torch.float32)
y_train_t = torch.tensor(y_train_np.reshape(-1, 1), dtype=torch.float32)
X_test_t = torch.tensor(X_test_np, dtype=torch.float32)
y_test_t = torch.tensor(y_test_np, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True
)

class MoonMLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=16):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.layers(x)  # raw logits

model = MoonMLP().to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
history = []

for epoch in range(100):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    history.append(total_loss / len(train_loader.dataset))

model.eval()
with torch.inference_mode():
    test_logits = model(X_test_t.to(device)).cpu()
    test_probs = torch.sigmoid(test_logits).numpy().ravel()
    test_pred = (test_probs >= 0.5).astype(int)

print("Accuracy:", round((test_pred == y_test_np).mean(), 3))
print(classification_report(y_test_np, test_pred, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test_np, test_pred))

plt.figure(figsize=(7, 3))
plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("Training BCE loss")
plt.title("MLP training loss")
plt.grid(True, alpha=0.3)
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test_np, test_pred)
plt.title("Held-out test confusion matrix")
plt.show()

`BCEWithLogitsLoss` expects raw logits. Apply sigmoid only when converting logits to probabilities for prediction. The scaler is fitted on training features only.

## 7. Save and reload the model

In [ ]:
checkpoint_path = "pytorch_moons_state.pt"
torch.save(model.state_dict(), checkpoint_path)

reloaded_model = MoonMLP().to(device)
state = torch.load(checkpoint_path, map_location=device, weights_only=True)
reloaded_model.load_state_dict(state)
reloaded_model.eval()

sample = X_test_t[:10].to(device)
model.eval()
with torch.inference_mode():
    original_output = model(sample)
    reloaded_output = reloaded_model(sample)

print("Outputs match:", torch.allclose(original_output, reloaded_output))

Save the `state_dict` rather than relying on a serialized model object. For a real application, also save model configuration, label mapping, and preprocessing parameters such as the fitted scaler.

## 8. CPU/GPU and common errors

In [ ]:
print("Selected device:", device)
# Pattern for a new model and batch:
example_model = nn.Linear(4, 2).to(device)
example_batch = torch.randn(8, 4, device=device)
example_output = example_model(example_batch)
print("Output shape:", example_output.shape)

# Common checks:
assert example_batch.device == next(example_model.parameters()).device
assert example_output.shape == (8, 2)

### Troubleshooting checklist

- **Shape mismatch:** print `features.shape`, `targets.shape`, and model output shape.
- **Wrong target dtype:** `CrossEntropyLoss` expects integer class indices with dtype `torch.long`.
- **Wrong output for loss:** `BCEWithLogitsLoss` expects logits, not sigmoid probabilities.
- **Device mismatch:** move both model and batches to the same `device`.
- **Loss does not improve:** inspect learning rate, target encoding, preprocessing, and gradient flow.
- **Evaluation appears inconsistent:** use `model.eval()` and `torch.inference_mode()`.

## 9. Experiment checklist

Try the following, changing one factor at a time:

1. Optimizer: SGD vs Adam.
2. Learning rate: `0.001`, `0.01`, `0.1`.
3. Hidden width: `8`, `16`, `32`.
4. Batch size: `16`, `32`, `64`.

Record training loss, test metrics, and any instability. For more reliable model selection, add a validation split and reserve the test set for final evaluation.

## Key takeaways

- Tensors are the core data structure; shapes and dtypes must match the task.
- Autograd computes gradients; optimizers update parameters.
- A training loop combines forward pass, loss, backpropagation, and parameter updates.
- Evaluation uses inference mode and held-out data.
- Save `state_dict` and the preprocessing/configuration needed to reproduce predictions.